In [1]:
import pandas as pd
import numpy as np
import scipy
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm
from pathlib import Path

from celloracle import motif_analysis as ma
import celloracle as co
co.__version__


'0.20.0'

In [2]:
os.chdir('/home/kl467102/thesis/')
dataset_name = "kim23-hm1"
refgenome = "hg19"
cicero_output = Path("results/cicero_output")  / dataset_name
celloracle_dir = Path("results/celloracle_output") / dataset_name


## Load scATAC peak data and peak connection data made with Cicero

In [3]:
# Load scATAC-seq peak list.
peaks = pd.read_csv(cicero_output / "all_peaks.csv", index_col=0)
peaks = peaks.x.values
peaks

array(['chr1_565094_565556', 'chr1_569170_569649', 'chr1_713565_714811',
       ..., 'chrY_28567009_28567482', 'chrY_28709168_28709483',
       'chrY_28817093_28817489'], dtype=object)

In [4]:
# Load Cicero coaccessibility scores.
cicero_connections = pd.read_csv(cicero_output / "cicero_connections.csv", index_col=0)
cicero_connections.head()

,Peak1,Peak2,coaccess
1,chr10_100018343_100018835,chr10_99788052_99788082,0.000000
2,chr10_100018343_100018835,chr10_99790128_99791108,-0.000179
3,chr10_100018343_100018835,chr10_99798032_99798496,0.000000
4,chr10_100018343_100018835,chr10_99801160_99801628,0.000000
5,chr10_100018343_100018835,chr10_99805468_99805886,0.000000


## Annotate transcription start sites

In [5]:
tss_annotated = ma.get_tss_info(peak_str_list=peaks, ref_genome=refgenome) 

# Check results
tss_annotated.tail()


que bed peaks: 142632
tss peaks in que: 21887


,chr,start,end,gene_short_name,strand
21882,chr12,14720270,14720937,PLBD1,-
21883,chr4,156679652,156681382,GUCY1B1,+
21884,chrX,132548997,132550114,GPC4,-
21885,chr7,100546685,100547742,MUC3A,+
21886,chr7,112727893,112728010,GPR85,-


## Integrate TSS info and cicero connections

In [6]:
integrated = ma.integrate_tss_peak_with_cicero(tss_peak=tss_annotated, 
                                               cicero_connections=cicero_connections)
print(integrated.shape)
integrated.head()


(356624, 3)


,peak_id,gene_short_name,coaccess
0,chr10_100027444_100030515,LOXL4,1.000000
1,chr10_100174489_100175328,PYROXD2,1.000000
2,chr10_100205811_100206960,CRTAC1,0.047436
3,chr10_100205811_100206960,HPS1,1.000000
4,chr10_100205811_100206960,LOC101927278,1.000000


### filter peaks

In [7]:
peak = integrated[integrated.coaccess >= 0.8]
peak = peak[["peak_id", "gene_short_name"]].reset_index(drop=True)
print(peak.shape)
peak.head()


(19054, 2)


,peak_id,gene_short_name
0,chr10_100027444_100030515,LOXL4
1,chr10_100174489_100175328,PYROXD2
2,chr10_100205811_100206960,HPS1
3,chr10_100205811_100206960,LOC101927278
4,chr10_101088941_101090070,CNNM1


In [8]:
peaks_path = cicero_output / "peak_gene_list.csv"
peak.to_csv(peaks_path)
